# **-- Visão Computacional: Comparativo de Abordagens --**

## 📌 Introdução

Dando continuidade ao projeto de visão computacional da FarmTech Solutions, esta entrega tem como objetivo comparar três abordagens distintas para detecção e classificação de objetos — especificamente **bananas** 🍌 e **laranjas** 🍊 — usando o mesmo dataset da Entrega 1.

As três abordagens avaliadas são:

1. **YOLOv5 customizado** *(Entrega 1)* — YOLOv5 treinado com o dataset próprio (resultados já obtidos)
2. **YOLO tradicional (pré-treinado)** — YOLOv5 padrão sem fine-tuning, usando pesos do COCO
3. **CNN treinada do zero** — Rede neural convolucional construída e treinada do início sobre o dataset

---

## 🎯 Objetivo

Avaliar criticamente cada abordagem em relação a:

- **Facilidade de uso/integração**
- **Precisão do modelo**
- **Tempo de treinamento/customização**
- **Tempo de inferência (predição)**

---

## 📂 Dataset

O dataset é o mesmo utilizado na Entrega 1:
- 80 imagens no total (40 bananas, 40 laranjas)
- Divisão: 64 treino | 8 validação | 8 teste
- **Link para download:** https://drive.google.com/drive/folders/1ONYyRtnXcSFOjG0KqdnXbfA3P3xczKVv?usp=sharing

> **Importante:** Faça o upload do dataset no seu Google Drive antes de executar este notebook. A estrutura esperada é:
> ```
> /content/drive/MyDrive/dataset_fase6/
>   train/images/
>   val/images/
>   test/images/
> ```

---
---

## ⚙️ Configuração do Ambiente

In [ ]:
# Montagem do Google Drive para acesso ao dataset
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Instalação e importação das bibliotecas necessárias
!pip install -q torch torchvision matplotlib scikit-learn seaborn

import os
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from IPython.display import display, Image as DisplayImage

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Verificação de GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo em uso: {device}')

In [ ]:
# Verificação de acesso ao dataset no Drive
base_path = '/content/drive/MyDrive/dataset_fase6'

for split in ['train', 'val', 'test']:
    path = os.path.join(base_path, split, 'images')
    files = os.listdir(path) if os.path.exists(path) else []
    print(f'{split}: {len(files)} imagens')

---
---

# 🔷 Abordagem 2 — YOLO Tradicional (Pré-treinado no COCO)

Nesta etapa, utilizamos o YOLOv5 **sem nenhum fine-tuning** — apenas os pesos pré-treinados no dataset COCO, que contém 80 classes gerais. O objetivo é avaliar se o modelo genérico já é capaz de reconhecer bananas e laranjas, e comparar seu desempenho ao modelo customizado da Entrega 1.

**Banana** (classe 46) e **laranja** (classe 49) existem no COCO, portanto o YOLO padrão deve detectá-las — mas sem a especialização do nosso dataset.

---

In [ ]:
# Download do repositório YOLOv5 e instalação das dependências
!git clone https://github.com/ultralytics/yolov5 yolov5_trad
%cd yolov5_trad
!pip install -q -r requirements.txt

In [ ]:
# Inferência com YOLO pré-treinado (pesos COCO — sem fine-tuning)
# Medição do tempo total de inferência

test_path = '/content/drive/MyDrive/dataset_fase6/test/images'

start_time = time.time()

!python detect.py \
    --weights yolov5s.pt \
    --img 640 \
    --conf 0.25 \
    --source {test_path} \
    --project runs/detect_trad \
    --name exp_coco \
    --exist-ok

elapsed = time.time() - start_time
print(f'\nTempo total de inferência (YOLO COCO): {elapsed:.2f}s para 8 imagens')
print(f'Média por imagem: {elapsed/8:.2f}s')

In [ ]:
# Visualização dos resultados do YOLO pré-treinado
result_dir = 'runs/detect_trad/exp_coco'
images = sorted(os.listdir(result_dir))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('YOLO Tradicional (COCO) — Resultados nas imagens de teste', fontsize=14, fontweight='bold')

for idx, (ax, img_file) in enumerate(zip(axes.flatten(), images)):
    img = Image.open(os.path.join(result_dir, img_file))
    ax.imshow(img)
    ax.set_title(img_file, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('yolo_trad_results.png', dpi=100, bbox_inches='tight')
plt.show()

### 📊 Análise — YOLO Tradicional

O YOLO pré-treinado no COCO detectou bananas e laranjas pois essas classes existem no dataset COCO (classes 46 e 49). No entanto, observam-se as seguintes diferenças em relação ao modelo customizado:

| Critério | YOLO Customizado (Entrega 1) | YOLO Tradicional (COCO) |
|---|---|---|
| **Precisão** | Alta (mAP50 ≈ 0.86 média) | Moderada — pode confundir variedades |
| **Treinamento** | ~10-15 min (45 épocas) | Nenhum — zero-shot |
| **Facilidade de uso** | Requer dataset + anotações | Plug-and-play, sem esforço |
| **Inferência** | ~270ms/imagem (CPU) | ~270ms/imagem (CPU) |
| **Especialização** | Adaptado ao nosso domínio | Genérico (80 classes COCO) |

> **Ponto-chave:** O YOLO genérico funciona razoavelmente bem para objetos presentes no COCO, mas perde precisão em variações específicas do nosso dataset (bananas cortadas, descascadas, diferentes iluminações), que o modelo customizado aprendeu durante o fine-tuning.

---
---

# 🔷 Abordagem 3 — CNN Treinada do Zero

Nesta etapa, construímos e treinamos uma **Rede Neural Convolucional (CNN)** do início, sem usar pesos pré-treinados. Diferente do YOLO (que detecta e localiza objetos via bounding boxes), a CNN aqui realiza **classificação de imagem inteira** — ou seja, responde: "esta imagem é uma banana ou uma laranja?"

A arquitetura foi projetada de forma progressiva:
- Blocos convolucionais para extração de features
- Pooling para redução dimensional
- Camadas totalmente conectadas para classificação
- Dropout para regularização e prevenção de overfitting

---

In [ ]:
# Voltar ao diretório raiz do Colab
import os
os.chdir('/content')

import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

In [ ]:
# Dataset customizado para carregar as imagens e inferir o label a partir do nome do arquivo
# Convenção: arquivos iniciados com 'tt-1' a 'tt-4' são bananas; 'tt-5' a 'tt-8' são laranjas
# Para treino/val, as imagens estão nas subpastas train/images e val/images
# O label é determinado pela pasta de origem (banana ou laranja) via mapeamento por nome de arquivo

class FruitDataset(Dataset):
    """Dataset que carrega imagens de banana e laranja.
    
    Como as imagens não estão em subpastas por classe, o label é inferido
    a partir dos arquivos de labels YOLO (.txt) presentes na pasta 'labels'.
    Classe 0 = banana, Classe 1 = laranja.
    """
    def __init__(self, images_dir, labels_dir, transform=None):
        self.transform = transform
        self.samples = []  # lista de (caminho_imagem, label)
        
        for img_file in sorted(os.listdir(images_dir)):
            if not img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            
            img_path = os.path.join(images_dir, img_file)
            label_file = os.path.splitext(img_file)[0] + '.txt'
            label_path = os.path.join(labels_dir, label_file)
            
            if not os.path.exists(label_path):
                continue
            
            # Lê a primeira linha do arquivo de label YOLO para obter a classe
            with open(label_path, 'r') as f:
                line = f.readline().strip()
            
            if not line:
                continue
            
            class_id = int(line.split()[0])  # 0=banana, 1=laranja
            self.samples.append((img_path, class_id))
        
        print(f'Carregadas {len(self.samples)} amostras de {images_dir}')
        bananas = sum(1 for _, c in self.samples if c == 0)
        laranjas = sum(1 for _, c in self.samples if c == 1)
        print(f'  Bananas: {bananas} | Laranjas: {laranjas}')
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [ ]:
# Transformações aplicadas às imagens
# Data augmentation no treino para aumentar a robustez com dataset pequeno
train_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Validação e teste sem augmentation
eval_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Caminhos do dataset
base = '/content/drive/MyDrive/dataset_fase6'

print('=== Treino ===')
train_dataset = FruitDataset(
    images_dir=f'{base}/train/images',
    labels_dir=f'{base}/train/labels',
    transform=train_transforms
)

print('\n=== Validação ===')
val_dataset = FruitDataset(
    images_dir=f'{base}/val/images',
    labels_dir=f'{base}/val/labels',
    transform=eval_transforms
)

print('\n=== Teste ===')
test_dataset = FruitDataset(
    images_dir=f'{base}/test/images',
    labels_dir=f'{base}/test/labels',
    transform=eval_transforms
)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=8, shuffle=False, num_workers=2)

In [ ]:
# Arquitetura da CNN construída do zero
class FruitCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(FruitCNN, self).__init__()
        
        # Bloco 1: extrai features de baixo nível (bordas, texturas)
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),      # 128x128 → 64x64
            nn.Dropout2d(0.1)
        )
        
        # Bloco 2: features de nível médio (formas, padrões de cor)
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),      # 64x64 → 32x32
            nn.Dropout2d(0.15)
        )
        
        # Bloco 3: features de alto nível (estrutura geral do objeto)
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),      # 32x32 → 16x16
            nn.Dropout2d(0.2)
        )
        
        # Pooling global para tornar a rede independente do tamanho de entrada
        self.global_avg_pool = nn.AdaptiveAvgPool2d((4, 4))
        
        # Classificador totalmente conectado
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.global_avg_pool(x)
        x = self.classifier(x)
        return x


model = FruitCNN(num_classes=2).to(device)

# Contagem de parâmetros treináveis
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parâmetros treináveis: {total_params:,}')
print(model)

In [ ]:
# Configuração de treinamento
criterion = nn.CrossEntropyLoss()  # perda para classificação multi-classe
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Reduz o LR se a val_loss parar de melhorar por 5 épocas
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5, verbose=True
)

EPOCHS = 50
CLASS_NAMES = ['banana', 'laranja']

print(f'Treinando por {EPOCHS} épocas | Dispositivo: {device}')

In [ ]:
# Loop de treinamento da CNN
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0

train_start = time.time()

for epoch in range(1, EPOCHS + 1):
    # --- Fase de treino ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss   += loss.item() * images.size(0)
        preds         = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total   += labels.size(0)
    
    # --- Fase de validação ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss    += loss.item() * images.size(0)
            preds        = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total   += labels.size(0)
    
    # Métricas da época
    t_loss = train_loss / train_total
    t_acc  = train_correct / train_total
    v_loss = val_loss / val_total
    v_acc  = val_correct / val_total
    
    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)
    
    scheduler.step(v_loss)
    
    # Salva o melhor modelo
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save(model.state_dict(), 'best_cnn_model.pth')
    
    if epoch % 5 == 0 or epoch == 1:
        print(f'Época {epoch:3d}/{EPOCHS} | '
              f'Train Loss: {t_loss:.4f}, Acc: {t_acc:.4f} | '
              f'Val Loss: {v_loss:.4f}, Acc: {v_acc:.4f}')

train_time = time.time() - train_start
print(f'\n✅ Treinamento concluído em {train_time:.1f}s ({train_time/60:.1f} min)')
print(f'Melhor acurácia de validação: {best_val_acc:.4f}')

In [ ]:
# Visualização da curva de aprendizado da CNN
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CNN do Zero — Curva de Aprendizado', fontsize=14, fontweight='bold')

epochs_range = range(1, EPOCHS + 1)

# Gráfico de perda (loss)
axes[0].plot(epochs_range, history['train_loss'], label='Treino', color='steelblue')
axes[0].plot(epochs_range, history['val_loss'],   label='Validação', color='coral')
axes[0].set_title('Loss (Perda)')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Gráfico de acurácia
axes[1].plot(epochs_range, history['train_acc'], label='Treino', color='steelblue')
axes[1].plot(epochs_range, history['val_acc'],   label='Validação', color='coral')
axes[1].set_title('Acurácia')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Acurácia')
axes[1].set_ylim([0, 1.05])
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cnn_learning_curve.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Avaliação da CNN no conjunto de teste
model.load_state_dict(torch.load('best_cnn_model.pth'))
model.eval()

all_preds, all_labels = [], []
inference_times = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        
        t0 = time.time()
        outputs = model(images)
        inference_times.append(time.time() - t0)
        
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

avg_inference = np.mean(inference_times) / len(test_dataset) * 1000  # ms por imagem

print('=== Resultados da CNN no Conjunto de Teste ===')
print(f'Acurácia: {accuracy_score(all_labels, all_preds):.4f}')
print(f'Tempo médio de inferência: {avg_inference:.1f}ms por imagem')
print()
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

In [ ]:
# Matriz de confusão da CNN
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('CNN do Zero — Matriz de Confusão (Teste)', fontweight='bold')
plt.ylabel('Classe Real')
plt.xlabel('Classe Predita')
plt.tight_layout()
plt.savefig('cnn_confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Visualização das predições da CNN nas imagens de teste
model.eval()
test_images_dir = f'{base}/test/images'
test_imgs = sorted([
    f for f in os.listdir(test_images_dir)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('CNN do Zero — Predições nas Imagens de Teste', fontsize=14, fontweight='bold')

for idx, (ax, img_file) in enumerate(zip(axes.flatten(), test_imgs)):
    img_path = os.path.join(test_images_dir, img_file)
    img = Image.open(img_path).convert('RGB')
    
    # Pré-processamento para inferência
    tensor = eval_transforms(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(tensor)
        prob = torch.softmax(output, dim=1)
        pred_class = prob.argmax(dim=1).item()
        confidence = prob[0][pred_class].item()
    
    true_label = all_labels[idx]
    pred_label = CLASS_NAMES[pred_class]
    true_name  = CLASS_NAMES[true_label]
    color      = 'green' if pred_class == true_label else 'red'
    
    ax.imshow(img)
    ax.set_title(f'Pred: {pred_label} ({confidence:.0%})\nReal: {true_name}',
                 color=color, fontsize=9, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('cnn_predictions.png', dpi=100, bbox_inches='tight')
plt.show()

---
---

# 📊 Comparativo Final das Três Abordagens

Nesta seção, consolidamos os resultados das três abordagens avaliadas ao longo das duas entregas.

---

In [ ]:
import pandas as pd

# Tabela comparativa consolidada
# Valores do YOLOv5 customizado vêm da Entrega 1 (45 épocas = melhor resultado)
# Valores do YOLO COCO e CNN são preenchidos após execução deste notebook

data = {
    'Critério': [
        'Precisão (Banana)',
        'Precisão (Laranja)',
        'Recall (Banana)',
        'Recall (Laranja)',
        'mAP50 / Acurácia Geral',
        'Tempo de treinamento',
        'Tempo de inferência (média)',
        'Facilidade de uso',
        'Necessita de anotações?',
    ],
    'YOLOv5 Customizado\n(Entrega 1, 45 épocas)': [
        '0.915',
        '0.607',
        '0.750',
        '1.000',
        'mAP50 ≈ 0.864',
        '~10-15 min (CPU)',
        '~270ms',
        'Média — requer anotações e configuração',
        'Sim (bounding boxes)',
    ],
    'YOLO Tradicional\n(COCO, zero-shot)': [
        'Ver saída do detect.py',
        'Ver saída do detect.py',
        'Ver saída do detect.py',
        'Ver saída do detect.py',
        'Moderado (classes COCO)',
        'Nenhum (zero-shot)',
        '~270ms',
        'Alta — plug-and-play, sem treino',
        'Não',
    ],
    'CNN do Zero': [
        'Ver classification_report',
        'Ver classification_report',
        'Ver classification_report',
        'Ver classification_report',
        'Ver accuracy_score',
        'Variável (~3-10 min)',
        '<10ms (só classificação)',
        'Baixa — requer arquitetura + treino',
        'Não (só label de classe)',
    ]
}

df = pd.DataFrame(data)
df = df.set_index('Critério')

print('=== TABELA COMPARATIVA DAS TRÊS ABORDAGENS ===')
print(df.to_string())

In [ ]:
# Gráfico comparativo de facilidade vs. precisão (radar/spider ou barras)
# Usamos barras lado a lado com pontuações qualitativas normalizadas de 0 a 5

criterios = ['Precisão\n(especialização)', 'Facilidade\nde uso', 'Velocidade de\nconfiguração',
             'Velocidade de\ninferência', 'Custo de\nanotação (inv.)']

# Pontuações subjetivas (0-5): refletem o comportamento esperado das abordagens
yolo_custom = [4.5, 3.0, 2.5, 3.5, 1.0]  # alta precisão, mas requer muito esforço
yolo_coco   = [3.0, 5.0, 5.0, 3.5, 5.0]  # zero esforço, mas menos especializado
cnn_zero    = [3.5, 2.0, 2.0, 5.0, 4.0]  # rápida inferência, muito trabalho de arquitetura

x = np.arange(len(criterios))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 6))
bars1 = ax.bar(x - width,     yolo_custom, width, label='YOLOv5 Customizado', color='steelblue',  alpha=0.85)
bars2 = ax.bar(x,             yolo_coco,   width, label='YOLO Tradicional',   color='coral',      alpha=0.85)
bars3 = ax.bar(x + width,     cnn_zero,    width, label='CNN do Zero',        color='mediumseagreen', alpha=0.85)

ax.set_ylabel('Pontuação (0-5)', fontsize=11)
ax.set_title('Comparativo de Abordagens — Avaliação por Critério', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(criterios, fontsize=10)
ax.set_ylim(0, 6)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{bar.get_height()}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('comparativo_abordagens.png', dpi=100, bbox_inches='tight')
plt.show()

---
---

## 🧠 Avaliação Crítica — Análise Comparativa

### 1. YOLOv5 Customizado (Entrega 1)

**Pontos fortes:**
- Maior especialização no domínio do problema — o modelo aprendeu especificamente as variações de banana e laranja do nosso dataset (cortadas, descascadas, inteiras)
- Realiza **detecção** (localização + classificação via bounding box), não apenas classificação
- Métricas robustas: mAP50 de 0.888 (banana) e 0.84 (laranja) com 45 épocas

**Limitações:**
- Exige um pipeline completo: coleta de imagens → anotação (bounding boxes) → configuração do YAML → treinamento
- O dataset pequeno (32 imagens de treino por classe) limita o potencial de generalização
- Sensível à qualidade das anotações: bounding boxes imprecisos afetaram diretamente o desempenho inicial

---

### 2. YOLO Tradicional — COCO (zero-shot)

**Pontos fortes:**
- **Facilidade máxima**: não requer treino, anotações ou configuração — apenas um comando
- Detecta objetos em tempo real com localização precisa
- Funciona razoavelmente para objetos presentes no COCO (banana = classe 46, laranja = classe 49)

**Limitações:**
- **Menor especialização**: o modelo genérico não foi exposto às variações específicas do nosso dataset (frutas em diferentes estados, ângulos e iluminações)
- Pode falhar em cenas muito diferentes das imagens COCO (fundos complexos, variações incomuns)
- Velocidade de inferência similar ao customizado — não há vantagem aqui

---

### 3. CNN Treinada do Zero

**Pontos fortes:**
- **Inferência extremamente rápida**: sem o overhead de detecção de objetos, a classificação é ordens de magnitude mais veloz
- Não requer anotações de bounding box — apenas o label da classe por imagem
- Totalmente customizável: arquitetura, hiperparâmetros e regularização sob controle total
- Aprendeu features específicas do dataset com data augmentation

**Limitações:**
- Realiza apenas **classificação**, não detecção — não localiza o objeto na imagem
- Com apenas 64 imagens de treino, o risco de overfitting é elevado mesmo com dropout e augmentation
- Requer maior conhecimento técnico para projetar e ajustar a arquitetura
- Sem transfer learning, o modelo precisa aprender todas as features do zero, o que é desafiador com poucos dados

---

### 📌 Conclusão Comparativa

| Cenário | Abordagem Recomendada |
|---|---|
| Prototipagem rápida, sem infraestrutura | YOLO Tradicional (COCO) |
| Detecção precisa em domínio específico | YOLOv5 Customizado |
| Classificação em tempo real / edge devices | CNN do Zero |
| Melhor custo-benefício com dados limitados | YOLOv5 Customizado |

Como demonstrado ao longo deste projeto, **não existe uma abordagem universalmente superior**. A escolha depende do cenário: disponibilidade de dados anotados, necessidade de localização vs. apenas classificação, tempo disponível para treinamento e requisitos de velocidade de inferência em produção.

No contexto da FarmTech Solutions — onde a precisão e a especialização no domínio são críticas — o **YOLOv5 customizado** apresentou a melhor relação entre esforço e qualidade dos resultados. Para sistemas embarcados ou aplicações que exijam resposta em milissegundos, a **CNN do zero** se torna a escolha natural, especialmente quando o escopo se limita à classificação.

---
---

## 🧠 Tecnologias Utilizadas

- Python
- PyTorch + Torchvision
- YOLOv5 (Ultralytics)
- Scikit-learn
- Matplotlib / Seaborn
- Google Colab + Google Drive